# Progres 2 (Minggu 13): Deskripsi Data

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = 30

import warnings
warnings.filterwarnings('ignore')

# folder gambar (opsional, untuk laporan)
IMG_DIR = '../Images' if os.path.isdir('../Data') else 'Images'
os.makedirs(IMG_DIR, exist_ok=True)
def simpan(nama):
    plt.savefig(os.path.join(IMG_DIR, nama), dpi=150, bbox_inches='tight')

## 1. Gambaran Umum Data

In [2]:
# Path otomatis: jalan baik dari folder Notebook/ maupun dari root repo
kandidat = ['../Data/pdb_industri_makanan_minuman_bps.csv',
            'Data/pdb_industri_makanan_minuman_bps.csv',
            'pdb_industri_makanan_minuman_bps.csv']
path = next(p for p in kandidat if os.path.exists(p))

df = pd.read_csv(path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values('tanggal').reset_index(drop=True)

print('File            :', path)
print('Ukuran dataset  :', df.shape[0], 'baris x', df.shape[1], 'kolom')
print('Periode         :', df['periode'].iloc[0], 's.d.', df['periode'].iloc[-1])
print('Frekuensi       : triwulanan (4 observasi per tahun)')
print()
df.info()

File            : ../Data/pdb_industri_makanan_minuman_bps.csv
Ukuran dataset  : 67 baris x 8 kolom
Periode         : 2010-Q1 s.d. 2026-Q3
Frekuensi       : triwulanan (4 observasi per tahun)

<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   periode                               67 non-null     str           
 1   tanggal                               67 non-null     datetime64[us]
 2   tahun                                 67 non-null     int64         
 3   triwulan                              67 non-null     int64         
 4   pdb_mamin_adhk_miliar_rp              67 non-null     float64       
 5   pdb_mamin_adhb_miliar_rp              67 non-null     float64       
 6   pdb_industri_nonmigas_adhk_miliar_rp  67 non-null     float64       
 7   hari_ramadan_idul_fitri               67 non-

In [3]:
df.head(8)

,periode,tanggal,tahun,triwulan,pdb_mamin_adhk_miliar_rp,pdb_mamin_adhb_miliar_rp,pdb_industri_nonmigas_adhk_miliar_rp,hari_ramadan_idul_fitri
0,2010-Q1,2010-01-01,2010,1,"85,184.00","84,636.90","311,629.30",0
1,2010-Q2,2010-04-01,2010,2,"88,613.60","87,305.40","318,492.00",0
2,2010-Q3,2010-07-01,2010,3,"93,661.20","93,999.20","324,139.80",32
3,2010-Q4,2010-10-01,2010,4,"92,984.30","94,501.60","324,677.50",0
4,2011-Q1,2011-01-01,2011,1,"94,319.50","96,680.80","330,619.30",0
5,2011-Q2,2011-04-01,2011,2,"100,242.20","99,883.60","343,015.20",0
6,2011-Q3,2011-07-01,2011,3,"102,360.80","106,741.80","348,918.40",32
7,2011-Q4,2011-10-01,2011,4,"103,081.20","107,081.20","351,847.20",0


In [4]:
df.tail(8)

,periode,tanggal,tahun,triwulan,pdb_mamin_adhk_miliar_rp,pdb_mamin_adhb_miliar_rp,pdb_industri_nonmigas_adhk_miliar_rp,hari_ramadan_idul_fitri
59,2024-Q4,2024-10-01,2024,4,"231,020.50","400,735.40","613,257.60",0
60,2025-Q1,2025-01-01,2025,1,"231,463.40","408,119.20","612,431.10",31
61,2025-Q2,2025-04-01,2025,2,"234,699.80","412,542.40","618,724.50",1
62,2025-Q3,2025-07-01,2025,3,"243,920.90","432,640.90","644,173.60",0
63,2025-Q4,2025-10-01,2025,4,"246,747.00","445,562.60","644,431.50",0
64,2026-Q1,2026-01-01,2026,1,"247,756.40","452,420.80","643,916.70",32
65,2026-Q2,2026-04-01,2026,2,"249,967.80","460,184.50","651,614.30",0
66,2026-Q3,2026-07-01,2026,3,"261,225.30","487,258.10","682,073.00",0


**Sumber data:** Badan Pusat Statistik (BPS) melalui Pusat Data dan Informasi (Pusdatin) Kementerian Perindustrian.
**Variabel target:** PDB Industri Makanan dan Minuman Atas Dasar Harga Konstan 2010 (`pdb_mamin_adhk_miliar_rp`), dalam Miliar Rupiah.

### Kamus variabel

In [5]:
kamus = pd.DataFrame([
    ['periode',  'Kategorikal', '-', 'Label periode, format TAHUN-Qn (mis. 2010-Q1)'],
    ['tanggal',  'Tanggal', '-', 'Tanggal awal triwulan; dipakai sebagai indeks waktu'],
    ['tahun',    'Numerik diskret', 'tahun', 'Tahun observasi (2010 s.d. 2026)'],
    ['triwulan', 'Numerik diskret', '1 s.d. 4', 'Nomor triwulan; dasar pola musiman'],
    ['pdb_mamin_adhk_miliar_rp', 'Numerik kontinu', 'Miliar Rp', 'TARGET: PDB Mamin harga konstan 2010 (pertumbuhan riil)'],
    ['pdb_mamin_adhb_miliar_rp', 'Numerik kontinu', 'Miliar Rp', 'PDB Mamin harga berlaku (nominal); pembanding'],
    ['pdb_industri_nonmigas_adhk_miliar_rp', 'Numerik kontinu', 'Miliar Rp', 'PDB Industri Pengolahan Non-Migas harga konstan 2010'],
    ['hari_ramadan_idul_fitri', 'Numerik diskret', 'hari', 'Jumlah hari Ramadan + Idul Fitri dalam triwulan; regressor kalender'],
], columns=['Kolom', 'Tipe', 'Satuan', 'Keterangan'])
kamus

,Kolom,Tipe,Satuan,Keterangan
0,periode,Kategorikal,-,"Label periode, format TAHUN-Qn (mis. 2010-Q1)"
1,tanggal,Tanggal,-,Tanggal awal triwulan; dipakai sebagai indeks ...
2,tahun,Numerik diskret,tahun,Tahun observasi (2010 s.d. 2026)
3,triwulan,Numerik diskret,1 s.d. 4,Nomor triwulan; dasar pola musiman
4,pdb_mamin_adhk_miliar_rp,Numerik kontinu,Miliar Rp,TARGET: PDB Mamin harga konstan 2010 (pertumbu...
5,pdb_mamin_adhb_miliar_rp,Numerik kontinu,Miliar Rp,PDB Mamin harga berlaku (nominal); pembanding
6,pdb_industri_nonmigas_adhk_miliar_rp,Numerik kontinu,Miliar Rp,PDB Industri Pengolahan Non-Migas harga konsta...
7,hari_ramadan_idul_fitri,Numerik diskret,hari,Jumlah hari Ramadan + Idul Fitri dalam triwula...


## 2. Kualitas Data
Pengecekan: missing value, duplikasi, nilai tidak wajar, kelengkapan urutan triwulan, dan konsistensi antar kolom waktu.

In [6]:
cek = pd.DataFrame({
    'Tipe data': df.dtypes.astype(str),
    'Missing': df.isnull().sum(),
    'Missing (%)': (df.isnull().mean() * 100).round(2),
    'Nilai unik': df.nunique(),
})
display(cek)
print('Jumlah baris duplikat       :', df.duplicated().sum())
print('Jumlah periode duplikat     :', df['periode'].duplicated().sum())

num_cols = ['pdb_mamin_adhk_miliar_rp', 'pdb_mamin_adhb_miliar_rp',
            'pdb_industri_nonmigas_adhk_miliar_rp', 'hari_ramadan_idul_fitri']
print('Nilai negatif di kolom PDB  :', int((df[num_cols[:3]] < 0).sum().sum()))
print('Nilai nol di kolom PDB      :', int((df[num_cols[:3]] == 0).sum().sum()))

,Tipe data,Missing,Missing (%),Nilai unik
periode,str,0,0.00,67
tanggal,datetime64[us],0,0.00,67
tahun,int64,0,0.00,17
triwulan,int64,0,0.00,4
pdb_mamin_adhk_miliar_rp,float64,0,0.00,67
pdb_mamin_adhb_miliar_rp,float64,0,0.00,67
pdb_industri_nonmigas_adhk_miliar_rp,float64,0,0.00,67
hari_ramadan_idul_fitri,int64,0,0.00,14


Jumlah baris duplikat       : 0
Jumlah periode duplikat     : 0
Nilai negatif di kolom PDB  : 0
Nilai nol di kolom PDB      : 0


In [7]:
# Kelengkapan urutan triwulan: bandingkan dengan deret kuartalan penuh
penuh = pd.date_range(df['tanggal'].min(), df['tanggal'].max(), freq='QS')
hilang = penuh.difference(df['tanggal'])

# Konsistensi kolom periode/tahun/triwulan terhadap tanggal
label_dari_tanggal = df['tanggal'].dt.year.astype(str) + '-Q' + df['tanggal'].dt.quarter.astype(str)
konsisten = (
    (label_dari_tanggal == df['periode']).all()
    and (df['tanggal'].dt.year == df['tahun']).all()
    and (df['tanggal'].dt.quarter == df['triwulan']).all()
)

print('Jumlah triwulan seharusnya  :', len(penuh))
print('Jumlah triwulan di data     :', len(df))
print('Triwulan yang hilang        :', list(hilang) if len(hilang) else 'tidak ada')
print('Kolom waktu konsisten       :', konsisten)
print('Tahun lengkap (4 triwulan)  :', int((df.groupby('tahun').size() == 4).sum()), 'tahun')
print('Tahun tidak lengkap         :', df.groupby('tahun').size()[lambda s: s < 4].to_dict())

Jumlah triwulan seharusnya  : 67
Jumlah triwulan di data     : 67
Triwulan yang hilang        : tidak ada
Kolom waktu konsisten       : True
Tahun lengkap (4 triwulan)  : 16 tahun
Tahun tidak lengkap         : {2026: 3}


Data tidak memiliki missing value maupun duplikasi, urutan triwulan lengkap tanpa celah, dan kolom waktu konsisten satu sama lain. Satu catatan: **2026 baru memuat Q1 s.d. Q3**, sehingga perbandingan tahunan 2026 harus diperlakukan sebagai tahun parsial.

## 3. Statistik Deskriptif

In [8]:
def ringkas(s):
    return pd.Series({
        'N': s.count(),
        'Mean': s.mean(),
        'Std': s.std(),
        'CV (%)': s.std() / s.mean() * 100 if s.mean() != 0 else np.nan,
        'Min': s.min(),
        'Q1': s.quantile(0.25),
        'Median': s.median(),
        'Q3': s.quantile(0.75),
        'Max': s.max(),
        'Rentang': s.max() - s.min(),
        'Skewness': s.skew(),
        'Kurtosis': s.kurt(),
    })

stat = df[num_cols].apply(ringkas).T
stat

,N,Mean,Std,CV (%),Min,Q1,Median,Q3,Max,Rentang,Skewness,Kurtosis
pdb_mamin_adhk_miliar_rp,67.00,"166,771.15","48,653.10",29.17,"85,184.00","123,047.40","171,813.30","204,285.30","261,225.30","176,041.30",0.04,-1.14
pdb_mamin_adhb_miliar_rp,67.00,"236,709.51","110,887.49",46.85,"84,636.90","135,712.45","231,521.00","309,645.50","487,258.10","402,621.20",0.48,-0.76
pdb_industri_nonmigas_adhk_miliar_rp,67.00,"480,595.52","96,260.41",20.03,"311,629.30","403,389.25","480,955.60","545,101.70","682,073.00","370,443.70",0.08,-0.85
hari_ramadan_idul_fitri,67.00,8.12,12.95,159.51,0.00,0.00,0.00,16.00,32.00,32.00,1.15,-0.52


In [9]:
y = df.set_index('tanggal')['pdb_mamin_adhk_miliar_rp']
i_min, i_max = y.idxmin(), y.idxmax()
print(f"Target (ADHK)  min : {y.min():,.1f} pada {df.loc[df['tanggal']==i_min,'periode'].item()}")
print(f"Target (ADHK)  max : {y.max():,.1f} pada {df.loc[df['tanggal']==i_max,'periode'].item()}")
print(f"Kenaikan total     : {y.iloc[-1]/y.iloc[0]:.2f}x dari {y.iloc[0]:,.1f} ke {y.iloc[-1]:,.1f}")
n_th = (len(y) - 1) / 4
print(f"CAGR (rata-rata pertumbuhan tahunan majemuk) : {((y.iloc[-1]/y.iloc[0])**(1/n_th)-1)*100:.2f}% per tahun")

Target (ADHK)  min : 85,184.0 pada 2010-Q1
Target (ADHK)  max : 261,225.3 pada 2026-Q3
Kenaikan total     : 3.07x dari 85,184.0 ke 261,225.3
CAGR (rata-rata pertumbuhan tahunan majemuk) : 7.03% per tahun


**Catatan membaca tabel:** Nilai CV, skewness, dan kurtosis pada level deret yang bertren naik terutama mencerminkan *rentang tren*, bukan variasi acak. Karena itu deskripsi tambahan pada bagian 4 dan 5 (per tahun, per triwulan, dan laju pertumbuhan) lebih informatif untuk deret waktu.

## 4. Deskripsi Deret Waktu

### 4.1 Pola keseluruhan

In [10]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df['tanggal'], df['pdb_mamin_adhk_miliar_rp'], marker='o', ms=3, label='PDB Mamin ADHK (target)')
axes[0].plot(df['tanggal'], df['pdb_mamin_adhb_miliar_rp'], marker='o', ms=3, label='PDB Mamin ADHB (nominal)')
axes[0].set_title('PDB Industri Makanan dan Minuman: Harga Konstan vs Harga Berlaku')
axes[0].set_ylabel('Miliar Rupiah')
axes[0].legend()

axes[1].plot(df['tanggal'], df['pdb_industri_nonmigas_adhk_miliar_rp'], color='tab:green', marker='o', ms=3)
axes[1].set_title('PDB Industri Pengolahan Non-Migas ADHK 2010')
axes[1].set_ylabel('Miliar Rupiah')
axes[1].set_xlabel('Periode')

# tandai periode pandemi COVID-19 (2020-Q2 s.d. 2020-Q4)
for ax in axes:
    ax.axvspan(pd.Timestamp('2020-04-01'), pd.Timestamp('2020-12-31'), color='grey', alpha=0.15)
axes[0].text(pd.Timestamp('2020-04-15'), axes[0].get_ylim()[1]*0.97, 'COVID-19', va='top', fontsize=9)
plt.tight_layout()
simpan('p2_01_deret_waktu_variabel.png')
plt.show()

### 4.2 Ringkasan per tahun

In [11]:
def kuartal_ke_tahunan(g):
    return pd.Series({
        'Jumlah triwulan': len(g),
        'Rata-rata ADHK': g['pdb_mamin_adhk_miliar_rp'].mean(),
        'Total ADHK': g['pdb_mamin_adhk_miliar_rp'].sum(),
        'Min ADHK': g['pdb_mamin_adhk_miliar_rp'].min(),
        'Max ADHK': g['pdb_mamin_adhk_miliar_rp'].max(),
        'Rata-rata ADHB': g['pdb_mamin_adhb_miliar_rp'].mean(),
    })

tahunan = df.groupby('tahun').apply(kuartal_ke_tahunan)
tahunan['Jumlah triwulan'] = tahunan['Jumlah triwulan'].astype(int)

# growth tahunan hanya bermakna jika kedua tahun lengkap
lengkap = tahunan['Jumlah triwulan'] == 4
tahunan['Growth tahunan (%)'] = np.where(
    lengkap & lengkap.shift(1, fill_value=False),
    tahunan['Total ADHK'].pct_change() * 100, np.nan)
tahunan

,Jumlah triwulan,Rata-rata ADHK,Total ADHK,Min ADHK,Max ADHK,Rata-rata ADHB,Growth tahunan (%)
tahun,,,,,,,
2010,4,"90,110.77","360,443.10","85,184.00","93,661.20","90,110.77",NaN
2011,4,"100,000.93","400,003.70","94,319.50","103,081.20","102,596.85",10.98
2012,4,"110,335.43","441,341.70","103,036.40","114,593.70","114,443.35",10.33
2013,4,"114,820.75","459,283.00","107,293.30","119,115.70","122,785.60",4.07
2014,4,"125,714.05","502,856.20","117,397.30","129,518.30","140,504.15",9.49
2015,4,"135,189.10","540,756.40","126,979.10","138,564.10","161,767.97",7.54
2016,4,"146,446.58","585,786.30","136,633.80","152,327.30","185,202.55",8.33
2017,4,"159,958.60","639,834.40","147,160.70","167,236.10","208,606.28",9.23
2018,4,"172,615.62","690,462.50","165,954.90","179,341.30","231,860.88",7.91


In [12]:
th = tahunan[lengkap]
print(f"Rata-rata growth tahunan (tahun lengkap) : {th['Growth tahunan (%)'].mean():.2f}%")
print(f"Growth tahunan tertinggi : {th['Growth tahunan (%)'].max():.2f}% pada {int(th['Growth tahunan (%)'].idxmax())}")
print(f"Growth tahunan terendah  : {th['Growth tahunan (%)'].min():.2f}% pada {int(th['Growth tahunan (%)'].idxmin())}")

fig, ax = plt.subplots(figsize=(12, 4.5))
g = th['Growth tahunan (%)'].dropna()
ax.bar(g.index.astype(int), g.values, color=np.where(g.values >= 0, 'tab:blue', 'tab:red'))
ax.axhline(g.mean(), color='black', ls='--', lw=1, label=f'Rata-rata {g.mean():.2f}%')
ax.set_title('Pertumbuhan Tahunan PDB Mamin ADHK (tahun lengkap)')
ax.set_ylabel('%'); ax.set_xticks(g.index.astype(int)); ax.legend()
plt.tight_layout(); simpan('p2_02_growth_tahunan.png'); plt.show()

Rata-rata growth tahunan (tahun lengkap) : 6.76%
Growth tahunan tertinggi : 10.98% pada 2011
Growth tahunan terendah  : 1.58% pada 2020


### 4.3 Ringkasan per triwulan (pola musiman)

In [13]:
# Rasio terhadap tren (rata-rata bergerak terpusat 2x4) agar pola musiman tidak tercampur tren
ma = y.rolling(4).mean().rolling(2).mean().shift(-2)
df['rasio_tren'] = (y / ma).values

per_q = df.groupby('triwulan').agg(
    N=('pdb_mamin_adhk_miliar_rp', 'count'),
    Mean=('pdb_mamin_adhk_miliar_rp', 'mean'),
    Std=('pdb_mamin_adhk_miliar_rp', 'std'),
    Min=('pdb_mamin_adhk_miliar_rp', 'min'),
    Max=('pdb_mamin_adhk_miliar_rp', 'max'),
    Rasio_terhadap_tren=('rasio_tren', 'mean'),
)
per_q['Rasio_terhadap_tren'] = per_q['Rasio_terhadap_tren'].round(4)
per_q

,N,Mean,Std,Min,Max,Rasio_terhadap_tren
triwulan,,,,,,
1,17,"160,791.95","50,556.39","85,184.00","247,756.40",0.98
2,17,"167,420.42","48,829.71","88,613.60","249,967.80",1.01
3,17,"173,318.24","51,458.23","93,661.20","261,225.30",1.02
4,16,"165,477.92","47,221.45","92,984.30","246,747.00",0.99


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot = df.pivot(index='triwulan', columns='tahun', values='pdb_mamin_adhk_miliar_rp')
pivot.plot(ax=axes[0], marker='o', ms=3, colormap='viridis')
axes[0].set_title('Seasonal Plot per Triwulan, per Tahun')
axes[0].set_xticks([1, 2, 3, 4]); axes[0].set_ylabel('Miliar Rupiah')
axes[0].legend(title='Tahun', ncol=3, fontsize=7)

sns.boxplot(x='triwulan', y='rasio_tren', data=df.dropna(subset=['rasio_tren']), ax=axes[1])
axes[1].axhline(1, color='black', lw=0.8, ls='--')
axes[1].set_title('Rasio terhadap Tren per Triwulan (1 = sama dengan tren)')
axes[1].set_ylabel('Rasio')

plt.tight_layout(); simpan('p2_03_pola_per_triwulan.png'); plt.show()

Rasio terhadap tren di atas 1 berarti triwulan tersebut cenderung berada di atas tren, di bawah 1 berarti di bawah tren. Ini memberi gambaran pola musiman kuartalan sebelum model dibangun.

## 5. Pertumbuhan q-to-q dan y-on-y

In [15]:
df['growth_qtq'] = df['pdb_mamin_adhk_miliar_rp'].pct_change() * 100
df['growth_yoy'] = df['pdb_mamin_adhk_miliar_rp'].pct_change(4) * 100

gt = df[['growth_qtq', 'growth_yoy']].agg(['count', 'mean', 'std', 'min', 'median', 'max', 'skew']).T
display(gt)

print('Triwulan pertumbuhan q-to-q negatif :', int((df['growth_qtq'] < 0).sum()), 'dari', int(df['growth_qtq'].count()))
print('Triwulan pertumbuhan y-on-y negatif :', int((df['growth_yoy'] < 0).sum()), 'dari', int(df['growth_yoy'].count()))
print()
print('Rata-rata growth q-to-q per triwulan (%):')
print(df.groupby('triwulan')['growth_qtq'].mean().round(2).to_string())

,count,mean,std,min,median,max,skew
growth_qtq,66.00,1.77,3.37,-6.02,1.40,9.65,0.13
growth_yoy,63.00,6.78,3.08,0.22,6.81,13.77,0.03


Triwulan pertumbuhan q-to-q negatif : 19 dari 66
Triwulan pertumbuhan y-on-y negatif : 0 dari 63

Rata-rata growth q-to-q per triwulan (%):
triwulan
1   -0.14
2    4.81
3    3.40
4   -1.30


In [16]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
qq = df.set_index('tanggal')['growth_qtq'].dropna()
yy = df.set_index('tanggal')['growth_yoy'].dropna()

axes[0].bar(qq.index, qq, width=70, color=np.where(qq >= 0, 'tab:green', 'tab:red'))
axes[0].axhline(0, color='black', lw=0.8); axes[0].set_title('Pertumbuhan q-to-q (%)')

axes[1].plot(yy.index, yy, marker='o', ms=3, color='tab:purple')
axes[1].axhline(0, color='black', lw=0.8); axes[1].axhline(yy.mean(), color='grey', ls='--', lw=1, label=f'Rata-rata {yy.mean():.2f}%')
axes[1].set_title('Pertumbuhan y-on-y (%)'); axes[1].legend()

for ax in axes:
    ax.axvspan(pd.Timestamp('2020-04-01'), pd.Timestamp('2020-12-31'), color='grey', alpha=0.15)
plt.tight_layout(); simpan('p2_04_growth_qtq_yoy.png'); plt.show()

In [17]:
# Triwulan dengan pertumbuhan y-on-y terendah dan tertinggi
tampil = ['periode', 'pdb_mamin_adhk_miliar_rp', 'growth_qtq', 'growth_yoy']
print('5 pertumbuhan y-on-y terendah:')
display(df.nsmallest(5, 'growth_yoy')[tampil])
print('5 pertumbuhan y-on-y tertinggi:')
display(df.nlargest(5, 'growth_yoy')[tampil])

5 pertumbuhan y-on-y terendah:


,periode,pdb_mamin_adhk_miliar_rp,growth_qtq,growth_yoy
41,2020-Q2,"187,625.10",1.87,0.22
42,2020-Q3,"195,564.70",4.23,0.66
47,2021-Q4,"190,870.30",-5.69,1.23
43,2020-Q4,"188,550.40",-3.59,1.66
44,2021-Q1,"188,685.90",0.07,2.45


5 pertumbuhan y-on-y tertinggi:


,periode,pdb_mamin_adhk_miliar_rp,growth_qtq,growth_yoy
31,2017-Q4,"167,236.10",0.80,13.77
5,2011-Q2,"100,242.20",6.28,13.12
32,2018-Q1,"165,954.90",-0.77,12.77
10,2012-Q3,"114,593.70",4.61,11.95
17,2014-Q2,"127,729.10",8.80,10.86


## 6. Distribusi Variabel Target

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(y, kde=True, bins=15, ax=axes[0])
axes[0].set_title('Histogram PDB Mamin ADHK'); axes[0].set_xlabel('Miliar Rupiah')

sns.boxplot(x=y, ax=axes[1], color='lightsteelblue')
axes[1].set_title('Boxplot PDB Mamin ADHK'); axes[1].set_xlabel('Miliar Rupiah')

stats.probplot(y, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot terhadap Normal')

plt.tight_layout(); simpan('p2_05_distribusi_target.png'); plt.show()

sw_stat, sw_p = stats.shapiro(y)
print(f'Shapiro-Wilk: W = {sw_stat:.4f}, p-value = {sw_p:.4f}')
print(f'Skewness = {y.skew():.3f} | Kurtosis = {y.kurt():.3f}')
print('Catatan: uji ini dipakai hanya sebagai deskripsi bentuk sebaran level data. Pada deret waktu')
print('observasi saling bergantung, sehingga normalitas level tidak menjadi asumsi model ARIMA/HW.')

Shapiro-Wilk: W = 0.9579, p-value = 0.0234
Skewness = 0.039 | Kurtosis = -1.138
Catatan: uji ini dipakai hanya sebagai deskripsi bentuk sebaran level data. Pada deret waktu
observasi saling bergantung, sehingga normalitas level tidak menjadi asumsi model ARIMA/HW.


In [19]:
# Deteksi outlier pada level (IQR) dan pada growth q-to-q (Robust Z-Score)
q1, q3 = y.quantile([0.25, 0.75]); iqr = q3 - q1
out_level = y[(y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)]
print('Outlier level (aturan IQR 1,5):', len(out_level))

g = df['growth_qtq'].dropna()
med = g.median(); mad = np.median(np.abs(g - med))
rz = 0.6745 * (g - med) / mad
out_g = df.loc[rz[rz.abs() > 3.5].index, ['periode', 'growth_qtq']].assign(robust_z=rz[rz.abs() > 3.5].round(2))
print('Outlier growth q-to-q (|Robust Z| > 3,5):', len(out_g))
out_g

Outlier level (aturan IQR 1,5): 0
Outlier growth q-to-q (|Robust Z| > 3,5): 0


,periode,growth_qtq,robust_z


## 7. Variabel Kalender Hari Raya
`hari_ramadan_idul_fitri` mencatat berapa hari Ramadan dan Idul Fitri jatuh dalam tiap triwulan. Karena Ramadan maju sekitar 11 hari tiap tahun (kalender Hijriah), triwulan yang memuatnya bergeser dari waktu ke waktu. Inilah alasan variabel ini dipakai sebagai regressor di RegARIMA.

In [20]:
hr = df['hari_ramadan_idul_fitri']
print('Distribusi nilai variabel hari raya:')
print(hr.describe().round(2).to_string())
print()
print('Triwulan dengan hari raya > 0 :', int((hr > 0).sum()), 'dari', len(hr))
print('Triwulan dengan hari raya = 0 :', int((hr == 0).sum()))
print()
print('Jumlah triwulan berisi hari raya, menurut nomor triwulan:')
print(df[hr > 0].groupby('triwulan').size().reindex([1, 2, 3, 4], fill_value=0).to_string())

Distribusi nilai variabel hari raya:
count   67.00
mean     8.12
std     12.95
min      0.00
25%      0.00
50%      0.00
75%     16.00
max     32.00

Triwulan dengan hari raya > 0 : 23 dari 67
Triwulan dengan hari raya = 0 : 44

Jumlah triwulan berisi hari raya, menurut nomor triwulan:
triwulan
1     4
2    12
3     7
4     0


In [21]:
# Triwulan "dominan" Ramadan: triwulan dengan hari raya terbanyak di tiap tahun
dom = df.loc[df.groupby('tahun')['hari_ramadan_idul_fitri'].idxmax(), ['tahun', 'periode', 'triwulan', 'hari_ramadan_idul_fitri']]
dom = dom.rename(columns={'triwulan': 'Triwulan dominan', 'hari_ramadan_idul_fitri': 'Hari'}).set_index('tahun')

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df['tanggal'], y.values, marker='o', ms=3, color='tab:blue')
axes[0].set_title('PDB Mamin ADHK'); axes[0].set_ylabel('Miliar Rupiah')

axes[1].bar(df['tanggal'], hr, width=70, color='tab:orange')
axes[1].set_title('Jumlah Hari Ramadan dan Idul Fitri per Triwulan'); axes[1].set_ylabel('Hari')
plt.tight_layout(); simpan('p2_06_hari_raya.png'); plt.show()

print('Triwulan dominan Ramadan per tahun:')
dom['Triwulan dominan'].map(lambda q: f'Q{q}').to_frame().T

Triwulan dominan Ramadan per tahun:


tahun,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Triwulan dominan,Q3,Q3,Q3,Q3,Q3,Q3,Q2,Q2,Q2,Q2,Q2,Q2,Q2,Q2,Q1,Q1,Q1


In [22]:
# Perbandingan rasio terhadap tren: triwulan dengan vs tanpa hari raya, dikontrol per nomor triwulan
d = df.dropna(subset=['rasio_tren']).copy()
d['Ada hari raya'] = np.where(d['hari_ramadan_idul_fitri'] > 0, 'Ya', 'Tidak')
d['Dominan (>=15 hari)'] = np.where(d['hari_ramadan_idul_fitri'] >= 15, 'Ya', 'Tidak')

tabel = d.pivot_table(index='triwulan', columns='Dominan (>=15 hari)', values='rasio_tren', aggfunc=['mean', 'count'])
display(tabel.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(x='triwulan', y='rasio_tren', hue='Dominan (>=15 hari)', data=d, ax=ax)
ax.axhline(1, color='black', lw=0.8, ls='--')
ax.set_title('Rasio terhadap Tren: Triwulan dengan vs tanpa Dominasi Ramadan')
plt.tight_layout(); simpan('p2_07_efek_hari_raya.png'); plt.show()

mean      count     
Dominan (>=15 hari) Tidak   Ya Tidak   Ya
triwulan                                 
1                    0.97 0.99 13.00 3.00
2                    1.01 1.01  7.00 8.00
3                    1.03 1.02 10.00 6.00
4                    0.99  NaN 16.00  NaN

Perbandingan dilakukan **per nomor triwulan** supaya efek musiman biasa tidak tercampur dengan efek hari raya.

**Temuan jujur:** selisih rasio terhadap tren antara triwulan "dominan Ramadan" dan yang bukan relatif kecil (sekitar 1-2 poin persen, dan arahnya tidak seragam antar triwulan; Q4 tidak pernah memuat hari raya). Jadi secara visual efek hari raya **tidak menonjol**, dan sebagian efeknya mungkin sudah terserap pola musiman dan tren. Ini masih deskriptif; signifikansi efek sebenarnya akan diuji lewat koefisien regressor di RegARIMA pada progres berikutnya.

## 8. Hubungan Antarvariabel

In [23]:
# Variabel turunan: implisit deflator dan pangsa Mamin terhadap Non-Migas
df['implisit_deflator'] = df['pdb_mamin_adhb_miliar_rp'] / df['pdb_mamin_adhk_miliar_rp'] * 100
df['pangsa_mamin_nonmigas'] = df['pdb_mamin_adhk_miliar_rp'] / df['pdb_industri_nonmigas_adhk_miliar_rp'] * 100

turunan = df[['implisit_deflator', 'pangsa_mamin_nonmigas']].agg(['mean', 'std', 'min', 'max']).T
display(turunan)
print(f"Pangsa Mamin terhadap Non-Migas (rata-rata) : {df['pangsa_mamin_nonmigas'].mean():.2f}%")
print(f"Pangsa pada periode terbaru ({df['periode'].iloc[-1]}) : {df['pangsa_mamin_nonmigas'].iloc[-1]:.2f}%")
print(f"Pangsa pada periode awal ({df['periode'].iloc[0]})  : {df['pangsa_mamin_nonmigas'].iloc[0]:.2f}%")

,mean,std,min,max
implisit_deflator,134.66,25.74,98.52,186.53
pangsa_mamin_nonmigas,34.04,3.58,27.34,39.08


Pangsa Mamin terhadap Non-Migas (rata-rata) : 34.04%
Pangsa pada periode terbaru (2026-Q3) : 38.30%
Pangsa pada periode awal (2010-Q1)  : 27.34%


In [24]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(df['tanggal'], df['pangsa_mamin_nonmigas'], color='tab:red', marker='o', ms=3)
axes[0].axhline(df['pangsa_mamin_nonmigas'].mean(), color='grey', ls='--', lw=1, label='Rata-rata')
axes[0].set_title('Pangsa PDB Mamin terhadap Industri Non-Migas (ADHK, %)'); axes[0].legend()

axes[1].plot(df['tanggal'], df['implisit_deflator'], color='tab:brown', marker='o', ms=3)
axes[1].set_title('Implisit Deflator PDB Mamin (ADHB / ADHK x 100)')
plt.tight_layout(); simpan('p2_08_pangsa_deflator.png'); plt.show()

In [25]:
# Korelasi pada LEVEL vs pada PERTUMBUHAN (y-on-y)
kol = ['pdb_mamin_adhk_miliar_rp', 'pdb_mamin_adhb_miliar_rp',
       'pdb_industri_nonmigas_adhk_miliar_rp', 'hari_ramadan_idul_fitri']
nama = ['Mamin ADHK', 'Mamin ADHB', 'Non-Migas ADHK', 'Hari Raya']

lvl = df[kol].corr(); lvl.index = lvl.columns = nama
yo = pd.DataFrame({
    'Mamin ADHK': df['pdb_mamin_adhk_miliar_rp'].pct_change(4),
    'Mamin ADHB': df['pdb_mamin_adhb_miliar_rp'].pct_change(4),
    'Non-Migas ADHK': df['pdb_industri_nonmigas_adhk_miliar_rp'].pct_change(4),
    'Hari Raya': df['hari_ramadan_idul_fitri'].diff(4),
}).dropna().corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(lvl, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Korelasi Pearson: LEVEL')
sns.heatmap(yo, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Korelasi Pearson: PERTUMBUHAN y-on-y\n(Hari Raya: selisih tahunan)')
plt.tight_layout(); simpan('p2_09_korelasi.png'); plt.show()

**Cara membaca:** Korelasi pada level antar-PDB sangat tinggi karena sama-sama bertren naik, sehingga bisa menyesatkan (*spurious*). Korelasi pada pertumbuhan lebih jujur menggambarkan keterkaitan sebenarnya. Variabel hari raya hampir tidak berkorelasi dengan level target karena efeknya bersifat periodik-bergeser, bukan linear terhadap level; efeknya baru terlihat setelah tren dan musiman dikontrol (bagian 7).

## 9. Ringkasan Temuan dan Implikasi Pemodelan

In [26]:
n_th_full = int((df.groupby('tahun').size() == 4).sum())
cagr = ((y.iloc[-1] / y.iloc[0]) ** (1 / ((len(y) - 1) / 4)) - 1) * 100
q_tinggi = per_q['Rasio_terhadap_tren'].idxmax(); q_rendah = per_q['Rasio_terhadap_tren'].idxmin()

ringkasan = f'''
RINGKASAN DESKRIPSI DATA
------------------------
1. Data      : {len(df)} observasi triwulanan, {df['periode'].iloc[0]} s.d. {df['periode'].iloc[-1]} ({n_th_full} tahun lengkap + 2026 parsial).
2. Kualitas  : missing = {int(cek['Missing'].sum())}, duplikasi = {int(df[kamus['Kolom']].duplicated().sum())}, triwulan hilang = {len(hilang)}.
3. Level     : target naik {y.iloc[-1]/y.iloc[0]:.2f}x ({y.iloc[0]:,.1f} -> {y.iloc[-1]:,.1f} Miliar Rp), CAGR {cagr:.2f}% per tahun.
4. Musiman   : rasio terhadap tren tertinggi di Q{q_tinggi} ({per_q.loc[q_tinggi,'Rasio_terhadap_tren']:.4f}), terendah di Q{q_rendah} ({per_q.loc[q_rendah,'Rasio_terhadap_tren']:.4f}).
5. Growth    : rata-rata y-on-y {df['growth_yoy'].mean():.2f}% (min {df['growth_yoy'].min():.2f}%, max {df['growth_yoy'].max():.2f}%).
6. Hari raya : {int((hr>0).sum())} dari {len(hr)} triwulan memuat Ramadan/Idul Fitri; nilai maksimum {int(hr.max())} hari.
7. Pangsa    : Mamin = {df['pangsa_mamin_nonmigas'].mean():.2f}% dari Non-Migas (rata-rata), {df['pangsa_mamin_nonmigas'].iloc[-1]:.2f}% pada {df['periode'].iloc[-1]}.
'''
print(ringkasan)


RINGKASAN DESKRIPSI DATA
------------------------
1. Data      : 67 observasi triwulanan, 2010-Q1 s.d. 2026-Q3 (16 tahun lengkap + 2026 parsial).
2. Kualitas  : missing = 0, duplikasi = 0, triwulan hilang = 0.
3. Level     : target naik 3.07x (85,184.0 -> 261,225.3 Miliar Rp), CAGR 7.03% per tahun.
4. Musiman   : rasio terhadap tren tertinggi di Q3 (1.0235), terendah di Q1 (0.9760).
5. Growth    : rata-rata y-on-y 6.78% (min 0.22%, max 13.77%).
6. Hari raya : 23 dari 67 triwulan memuat Ramadan/Idul Fitri; nilai maksimum 32 hari.
7. Pangsa    : Mamin = 34.04% dari Non-Migas (rata-rata), 38.30% pada 2026-Q3.



### Implikasi untuk tahap berikutnya

| Temuan deskriptif | Implikasi pemodelan |
|---|---|
| Tren naik jelas dan nilai riil meningkat >3x | Perlu differencing (atau komponen tren di Holt-Winters); konsisten dengan uji stasioneritas di Progres 1 |
| Pola musiman kuartalan; fluktuasi membesar seiring level | Pertimbangkan transformasi log (musiman multiplikatif) dan differencing musiman s = 4 |
| Posisi triwulan Ramadan/Idul Fitri bergeser (Q3 pada 2010-2015, Q2 pada 2016-2023, Q1 pada 2024-2026), tetapi efeknya pada rasio terhadap tren tampak kecil | Tetap masukkan `hari_ramadan_idul_fitri` sebagai regressor RegARIMA, lalu uji signifikansinya; jangan diasumsikan kuat sebelum diuji |
| Terdapat guncangan pada periode pandemi 2020 | Perlu diperhatikan sebagai outlier/perubahan tren (mis. dummy intervensi) saat pemodelan |
| Hanya 67 observasi | Data uji sebaiknya kecil (mis. 8 triwulan terakhir); hati-hati terhadap model terlalu kompleks |
| Data 2026 baru sampai Q3 | Jangan dipakai untuk menghitung growth tahunan 2026 |

**Catatan asumsi:** Efek hari raya dan efek pandemi di atas bersifat dugaan dari deskripsi data; pembuktiannya dilakukan di tahap RegARIMA dan evaluasi model.